In [1]:
# Google Colab drive mount disabled for local execution
print('Running locally. Using local relative paths.')


Running locally. Using local relative paths.


In [2]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict
import tensorflow as tf
import random

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

I0000 00:00:1781193660.369806   14960 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781193660.374559   14960 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781193660.549767   14960 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1781193664.098048   14960 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781193664.099971   14960 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
CROP = "winter_wheat"

# Oct (prev year) → June (current year)
MONTHS = [10, 11, 12, 1, 2, 3]  # simplified window

print("Crop:", CROP)
print("Months:", MONTHS)

Crop: winter_wheat
Months: [10, 11, 12, 1, 2, 3]


In [4]:
data = np.load("../processed/full_weather_9features_v2.npz", allow_pickle=True)

X_seq_full = data["X_seq_full"]
meta = data["meta"]

print(X_seq_full.shape)

(17927, 12, 12)


In [5]:
# May → October (index 4 to 9)
X_crop = X_seq_full[:, 0:6, :]

print("X_crop shape:", X_crop.shape)

X_crop shape: (17927, 6, 12)


In [6]:
# soybean-specific feature indices
selected_features_idx = [0, 1, 2, 3, 4, 5, 6, 11]

X_crop = X_crop[:, :, selected_features_idx]

print("Updated X_crop shape:", X_crop.shape)

Updated X_crop shape: (17927, 6, 8)


In [7]:
from tensorflow.keras.layers import Input
seq_input = Input(shape=(6, 8))

In [8]:
import pandas as pd
yield_df = pd.read_csv("../processed/WinterWheat_all_years.csv")
yield_df.columns = ['fips', 'year', 'yield']
yield_df['fips'] = yield_df['fips'].astype(str).str.zfill(5)
yield_df['year'] = yield_df['year'].astype(str)
yield_df['yield'] = pd.to_numeric(yield_df['yield'], errors='coerce')
yield_df = yield_df.dropna(subset=['yield'])


In [9]:
yield_df.columns

Index(['fips', 'year', 'yield'], dtype='object')

In [10]:
# ansi processing merged into CSV loader


In [11]:
# grouping merged into CSV loader


In [12]:
print(yield_df.head())
print("Shape:", yield_df.shape)
print("Unique FIPS:", yield_df['fips'].nunique())

    fips  year  yield
0  05003  2017   63.3
1  05037  2017   65.6
2  05069  2017   44.4
3  05077  2017   47.9
4  05079  2017   45.8
Shape: (5082, 3)
Unique FIPS: 1240


In [13]:
# Build a mapping from (fips, year) to calendar-year weather sequence
weather_map = {(fips, int(year)): X_seq_full[idx] for idx, (fips, year) in enumerate(meta)}

final_X = []
final_y = []
final_meta = []

for i, (fips, year) in enumerate(meta):
    yr = int(year)
    prev_year_weather = weather_map.get((fips, yr - 1))
    curr_year_weather = weather_map.get((fips, yr))
    
    if prev_year_weather is not None and curr_year_weather is not None:
        match = yield_df[
            (yield_df['fips'] == fips) &
            (yield_df['year'] == str(yr))
        ]
        
        if len(match) == 1:
            # Slice Oct-Dec (indices 9,10,11) of prev year, and Jan-Mar (indices 0,1,2) of current year
            prev_seq = prev_year_weather[9:12, selected_features_idx]
            curr_seq = curr_year_weather[0:3, selected_features_idx]
            seq = np.concatenate([prev_seq, curr_seq], axis=0)
            
            final_X.append(seq)
            final_y.append(match['yield'].values[0])
            final_meta.append((fips, str(yr)))

X = np.array(final_X)
y = np.array(final_y)

print(X.shape, y.shape)


(4049, 6, 8) (4049,)


In [14]:
# total rain
total_rain = X[:, :, 4].sum(axis=1)

# avg temp
avg_temp_season = X[:, :, 0:2].mean(axis=(1,2))

# cold stress (VERY IMPORTANT)
cold_stress = (X[:, :, 1] < 0).sum(axis=1)

# spring rain (Mar–May → index 2,3,4)
spring_rain = X[:, 2:5, 4].sum(axis=1)

# total GDD
gdd_total = X[:, :, 3].sum(axis=1)

# avg humidity
avg_humidity = X[:, :, 5].mean(axis=1)

# ET
et_total = X[:, :, 7].sum(axis=1)

In [15]:
from collections import defaultdict

fips_arr = np.array([f for f, y_ in final_meta])
year_arr = np.array([y_ for f, y_ in final_meta]).astype(int)

yield_map = defaultdict(dict)

for i in range(len(y)):
    yield_map[fips_arr[i]][year_arr[i]] = y[i]

yield_lag_1 = np.zeros(len(y))
yield_lag_2 = np.zeros(len(y))

for i in range(len(y)):
    f = fips_arr[i]
    yr = year_arr[i]

    yield_lag_1[i] = yield_map[f].get(yr - 1, 0)
    yield_lag_2[i] = yield_map[f].get(yr - 2, 0)

yield_trend = yield_lag_1 - yield_lag_2


In [16]:
X_extra = np.stack([
    total_rain,
    avg_temp_season,
    yield_trend,
    cold_stress,
    spring_rain,
    gdd_total,
    avg_humidity,
    et_total
], axis=1)

print(X_extra.shape)

(4049, 8)


In [17]:
from sklearn.preprocessing import LabelEncoder

# Standardize fips_arr as strings to avoid type mixing
fips_arr = np.array([str(int(float(f))).zfill(5) if isinstance(f, (int, float, np.number)) else str(f).zfill(5) for f in fips_arr])
le = LabelEncoder()
fips_encoded = le.fit_transform(fips_arr)


In [18]:
train_mask = year_arr < 2022
test_mask = year_arr == 2022

X_train = X[train_mask]
X_test = X[test_mask]

X_lag_train = X_extra[train_mask]
X_lag_test = X_extra[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

fips_train = fips_encoded[train_mask]
fips_test = fips_encoded[test_mask]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (3233, 6, 8)
Test: (816, 6, 8)


In [19]:
from sklearn.preprocessing import StandardScaler

# sequence
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(
    X_train.reshape(-1, X_train.shape[-1])
).reshape(X_train.shape)

X_test_scaled = scaler_X.transform(
    X_test.reshape(-1, X_test.shape[-1])
).reshape(X_test.shape)

# lag
lag_scaler = StandardScaler()
X_lag_train_scaled = lag_scaler.fit_transform(X_lag_train)
X_lag_test_scaled = lag_scaler.transform(X_lag_test)

# target
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1,1))
y_test_scaled = y_scaler.transform(y_test.reshape(-1,1))

In [20]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate, Dropout, Embedding, Flatten
from tensorflow.keras.optimizers import Adam

# sequence input
seq_input = Input(shape=(6, 8), name='seq_input') # Changed shape from (6, 12) to (12, 7)
x = LSTM(96)(seq_input)
x = Dropout(0.2)(x)
x = Dense(32, activation='relu')(x)

# lag input
lag_input = Input(shape=(8,), name='lag_input')
y_lag = Dense(16, activation='relu')(lag_input)

# FIPS input and embedding
fips_input = Input(shape=(1,), name='fips_input')
fips_emb = Embedding(input_dim=fips_encoded.max() + 1, output_dim=16)(fips_input)
fips_emb = Flatten()(fips_emb) # Flatten the embedding output to concatenate with other layers

# combine
combined = Concatenate()([x, y_lag, fips_emb])

z = Dense(32, activation='relu')(combined)
output = Dense(1)(z)

model = Model(inputs=[seq_input, lag_input, fips_input], outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='mse',
    metrics=['mae']
)

model.summary()

E0000 00:00:1781193700.029197   14960 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ seq_input           │ (None, 6, 8)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 96)        │     40,320 │ seq_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fips_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 96)        │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lag_input           │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 16)     │     18,576 │ fips_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      3,104 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        144 │ lag_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 16)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64)        │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_1[0][0],    │
│                     │                   │            │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 64,257 (251.00 KB)

 Trainable params: 64,257 (251.00 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-5
)

history = model.fit(
    [X_train_scaled, X_lag_train_scaled, fips_train], y_train_scaled,
    validation_data=([X_test_scaled, X_lag_test_scaled, fips_test], y_test_scaled),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    shuffle=False
)

Epoch 1/50


 1/51 ━━━━━━━━━━━━━━━━━━━━ 4:05 5s/step - loss: 1.0927 - mae: 0.8227

 6/51 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.3900 - mae: 1.1920

11/51 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.1313 - mae: 1.1056

16/51 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1.8886 - mae: 1.0308

...

38/51 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1551 - mae: 0.2785

41/51 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1521 - mae: 0.2764

45/51 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1484 - mae: 0.2739

50/51 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1444 - mae: 0.2710

51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.1124 - mae: 0.2502 - val_loss: 0.4682 - val_mae: 0.5390


In [22]:
from sklearn.metrics import r2_score

y_pred_scaled = model.predict([X_test_scaled, X_lag_test_scaled, fips_test])
y_pred = y_scaler.inverse_transform(y_pred_scaled)

r2 = r2_score(y_test, y_pred)

print("🔥 Winter Wheat R²:", r2)

 1/26 ━━━━━━━━━━━━━━━━━━━━ 7s 313ms/step

13/26 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step  

25/26 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step


🔥 Winter Wheat R²: 0.7255497867717292


In [23]:
model.save("../Wheat_lstm_model.keras")

In [24]:
import joblib

joblib.dump(scaler_X, "../wheat_x_scaler.pkl")
joblib.dump(lag_scaler, "../wheat_lag_scaler.pkl")
joblib.dump(y_scaler, "../wheat_y_scaler.pkl")
joblib.dump(le, "../wheat_fips_encoder.pkl")

['../wheat_fips_encoder.pkl']

In [25]:
import pandas as pd

history_df = pd.DataFrame(history.history)
history_df.to_csv('../wheat_model_training_history.csv', index=False)
print('Training history saved successfully.')

Training history saved successfully.


In [26]:
import json

config = {
    "crop": "winter_wheat",

    "months_used": [1, 2, 3, 4, 5, 6],

    "sequence_features": [
        {"index": 0, "name": "max_temp_C"},
        {"index": 1, "name": "min_temp_C"},
        {"index": 2, "name": "temp_range"},
        {"index": 3, "name": "gdd"},
        {"index": 4, "name": "precipitation"},
        {"index": 5, "name": "relative_humidity"},
        {"index": 6, "name": "radiation"},
        {"index": 7, "name": "evapotranspiration"}
    ],

    "engineered_features": {
        "total_rain": "sum(X[:, :, 4])",
        "avg_temp_season": "mean(X[:, :, 0:2])",
        "cold_stress": "count(X[:, :, 1] < 0)",
        "spring_rain": "sum(X[:, 2:5, 4])",
        "gdd_total": "sum(X[:, :, 3])",
        "avg_humidity": "mean(X[:, :, 5])",
        "et_total": "sum(X[:, :, 7])"
    },

    "lag_features": [
        "yield_lag_1",
        "yield_lag_2",
        "yield_trend",
        "total_rain",
        "avg_temp_season",
        "cold_stress",
        "spring_rain",
        "gdd_total",
        "avg_humidity",
        "et_total"
    ],

    "input_shape": {
        "sequence": [6, 8],
        "lag": 8
    },

    "notes": "Winter wheat uses Jan–June window. Cold stress and spring rainfall are key drivers."
}

with open('../wheat_config.json', 'w') as f:
    json.dump(config, f, indent=4)

print("Winter wheat config saved successfully.")

Winter wheat config saved successfully.


In [27]:
np.savez("../wheat_data.npz",
         X=X,
         X_extra=X_extra,
         y=y,
         fips=fips_arr,
         year=year_arr)